<a href="https://colab.research.google.com/github/ShaojieDong503/HAD5015-Project/blob/main/Phase_2_StationData_Exclu_Met.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
target_path = "/content/drive/MyDrive/ML project data/pm25_targets.pkl"

import pandas as pd
dic_target = pd.read_pickle(target_path)



In [ ]:
for k in dic_target.keys():
  print(k)
  print(dic_target[k])
  break

df_target_2010
       naps_id       lat      lon        date       pm25
50735    60104  45.43433 -75.6760  2010-01-01   9.500000
50736    60104  45.43433 -75.6760  2010-01-02   0.250000
50737    60104  45.43433 -75.6760  2010-01-03   1.125000
50738    60104  45.43433 -75.6760  2010-01-04   1.000000
50739    60104  45.43433 -75.6760  2010-01-05   0.875000
...        ...       ...      ...         ...        ...
65330    65401  44.15053 -77.3955  2010-12-27   0.833333
65331    65401  44.15053 -77.3955  2010-12-28   5.500000
65332    65401  44.15053 -77.3955  2010-12-29   9.041667
65333    65401  44.15053 -77.3955  2010-12-30  10.750000
65334    65401  44.15053 -77.3955  2010-12-31   8.125000

[8101 rows x 5 columns]


In [ ]:
import pandas as pd
import numpy as np
import re

def build_station_monthly_pm25_by_year(target_dict):
    wide_by_year = {}

    for key, df in target_dict.items():
        # --- Extract year from key like "df_target_2010" ---
        m = re.search(r"(19|20)\d{2}", str(key))
        if not m:
            raise ValueError(f"Could not extract year from key: {key}")
        year = int(m.group(0))

        if df is None or df.empty:
            cols = ["naps_id", "lat", "lon"] + [f"pm25_{i:02d}" for i in range(1, 13)]
            wide_by_year[year] = pd.DataFrame(columns=cols)
            continue

        d = df.copy()

        # --- Types ---
        d["date"] = pd.to_datetime(d["date"], errors="coerce")
        d = d.dropna(subset=["date"])

        # Safety: keep only that year
        d = d[d["date"].dt.year == year]

        # --- Month ---
        d["month"] = d["date"].dt.month

        # --- Station coords (robust in case of tiny variations) ---
        coords = (
            d.groupby("naps_id", as_index=False)
             .agg(lat=("lat", "median"), lon=("lon", "median"))
        )

        # --- Monthly mean per station ---
        monthly = (
            d.groupby(["naps_id", "month"], as_index=False)["pm25"]
             .mean()
        )

        # --- Pivot to wide (12 months) ---
        wide_pm25 = (
            monthly.pivot(index="naps_id", columns="month", values="pm25")
                  .rename(columns={m: f"pm25_{m:02d}" for m in range(1, 13)})
        )

        # ensure all 12 columns exist
        for mth in range(1, 13):
            col = f"pm25_{mth:02d}"
            if col not in wide_pm25.columns:
                wide_pm25[col] = np.nan

        wide_pm25 = wide_pm25[[f"pm25_{mth:02d}" for mth in range(1, 13)]].reset_index()

        # --- Merge coords + pm25 ---
        wide_df = coords.merge(wide_pm25, on="naps_id", how="left")
        wide_df = wide_df.sort_values("naps_id").reset_index(drop=True)

        wide_by_year[year] = wide_df

    # Optional combined dataset (stack years)
    combined_wide = pd.concat(
        [df.assign(year=year) for year, df in sorted(wide_by_year.items())],
        ignore_index=True
    )

    # Put year first
    combined_wide = combined_wide[["year", "naps_id", "lat", "lon"] + [f"pm25_{m:02d}" for m in range(1, 13)]]

    return wide_by_year, combined_wide





In [ ]:
# -----------------------
# Example usage
# -----------------------
wide_by_year, combined_wide = build_station_monthly_pm25_by_year(dic_target)

wide_by_year[2010].head()
combined_wide.head()

,year,naps_id,lat,lon,pm25_01,pm25_02,pm25_03,pm25_04,pm25_05,pm25_06,pm25_07,pm25_08,pm25_09,pm25_10,pm25_11,pm25_12
0,2010,60104,45.43433,-75.67600,4.537256,1.907738,2.998399,3.940278,5.939516,5.134722,8.060730,7.258065,4.262198,2.862611,3.488889,3.289132
1,2010,60106,45.38287,-75.71387,3.851018,2.001488,3.123289,3.758696,5.956208,5.101800,8.118168,6.581989,3.671629,2.642898,3.712500,3.073983
2,2010,60204,42.31578,-83.04367,5.998656,4.608487,7.252653,6.844845,7.202392,9.086528,13.259124,13.658507,6.503436,5.534808,6.302041,5.532258
3,2010,60211,42.29289,-83.07314,5.888961,5.253923,7.525888,6.550821,7.826634,9.347503,13.613167,13.950845,6.117331,5.271055,6.852778,5.137097
4,2010,60303,44.22008,-76.52141,6.281884,3.226190,5.417113,5.041304,6.071616,9.090217,13.260414,11.346774,5.873322,3.284096,4.370864,4.647727


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations_pm25_combin.csv'
combined_wide.to_csv(output_path, index=False)

In [ ]:
wide_by_year[2010]

,naps_id,lat,lon,pm25_01,pm25_02,pm25_03,pm25_04,pm25_05,pm25_06,pm25_07,pm25_08,pm25_09,pm25_10,pm25_11,pm25_12
0,60104,45.43433,-75.67600,4.537256,1.907738,2.998399,3.940278,5.939516,5.134722,8.060730,7.258065,4.262198,2.862611,3.488889,3.289132
1,60106,45.38287,-75.71387,3.851018,2.001488,3.123289,3.758696,5.956208,5.101800,8.118168,6.581989,3.671629,2.642898,3.712500,3.073983
2,60204,42.31578,-83.04367,5.998656,4.608487,7.252653,6.844845,7.202392,9.086528,13.259124,13.658507,6.503436,5.534808,6.302041,5.532258
3,60211,42.29289,-83.07314,5.888961,5.253923,7.525888,6.550821,7.826634,9.347503,13.613167,13.950845,6.117331,5.271055,6.852778,5.137097
4,60303,44.22008,-76.52141,6.281884,3.226190,5.417113,5.041304,6.071616,9.090217,13.260414,11.346774,5.873322,3.284096,4.370864,4.647727
5,60410,43.74792,-79.27406,5.580933,3.537138,5.378915,5.917883,6.785531,7.676627,13.226352,12.250839,5.849715,4.643206,5.049242,4.577367
6,60413,43.64852,-79.59138,NaN,NaN,NaN,5.657773,7.133861,7.749440,10.422417,9.400161,5.034908,3.921576,5.654968,4.603057
7,60421,43.78158,-79.41773,5.043383,2.357996,4.953366,5.157126,6.705361,8.131209,11.227551,10.479079,5.394886,4.409946,5.611490,4.566445
8,60430,43.70944,-79.54350,5.493037,3.442096,5.449451,6.026216,7.338796,8.015366,11.437585,10.079593,5.253019,4.306917,5.929018,4.892707
9,60433,43.66417,-79.38722,3.819393,2.843963,4.728378,5.019790,5.952523,7.481039,10.641353,10.574201,6.582669,4.517824,5.218318,4.907258


In [ ]:
combined_wide.isna().sum()

,0
year,0
naps_id,0
lat,0
lon,0
pm25_01,4
pm25_02,4
pm25_03,5
pm25_04,3
pm25_05,4
pm25_06,2


In [ ]:
for y in range(2010,2024):
    output_path = f'/content/drive/MyDrive/ML project data/stations_pm25_{y}.csv'
    wide_by_year[y].to_csv(output_path, index=False)
    print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2010.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2011.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2012.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2013.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2014.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2015.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2016.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2017.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2018.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2019.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2020.csv
DataFrame saved to: /content/drive/MyDrive/ML project data/stations_pm25_2021.csv
DataFrame saved 

In [ ]:
def get_stations_df(wide_by_year, id_col="naps_id", lat_col="lat", lon_col="lon"):
    coords_all = pd.concat(
        [
            df[[id_col, lat_col, lon_col]].copy()
            for df in wide_by_year.values()
            if df is not None and not df.empty
        ],
        ignore_index=True
    ).dropna(subset=[id_col, lat_col, lon_col])

    # robust: if tiny float differences across years, take median
    stations_df = (
        coords_all.groupby(id_col, as_index=False)
                  .agg(**{lat_col: (lat_col, "median"),
                          lon_col: (lon_col, "median")})
    )

    # optional: stable rounding
    stations_df[lat_col] = stations_df[lat_col].round(6)
    stations_df[lon_col] = stations_df[lon_col].round(6)

    return stations_df.sort_values(id_col).reset_index(drop=True)

stations_df = get_stations_df(wide_by_year)
display(stations_df.head())
print("Unique stations:", stations_df.shape[0])

,naps_id,lat,lon
0,60104,45.43433,-75.67600
1,60106,45.38287,-75.71387
2,60204,42.31578,-83.04367
3,60211,42.29289,-83.07314
4,60303,44.22008,-76.52141


Unique stations: 34


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations.csv'
stations_df.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations.csv


In [ ]:
import os

folder = "/content/drive/MyDrive/ML project data/ontario_npz"

files = sorted([
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith(".npz")
])

print("Total files:", len(files))
print(files[:5])

Total files: 168
['/content/drive/MyDrive/ML project data/ontario_npz/V5NA05.HybridPM25.NorthAmerica.2010-2010-APR_ON.npz', '/content/drive/MyDrive/ML project data/ontario_npz/V5NA05.HybridPM25.NorthAmerica.2010-2010-AUG_ON.npz', '/content/drive/MyDrive/ML project data/ontario_npz/V5NA05.HybridPM25.NorthAmerica.2010-2010-DEC_ON.npz', '/content/drive/MyDrive/ML project data/ontario_npz/V5NA05.HybridPM25.NorthAmerica.2010-2010-FEB_ON.npz', '/content/drive/MyDrive/ML project data/ontario_npz/V5NA05.HybridPM25.NorthAmerica.2010-2010-JAN_ON.npz']


In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
from affine import Affine
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree

def _read_npz_meta(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    arr = d["arr"]
    nodata = d["nodata"].item() if np.ndim(d["nodata"]) == 0 else d["nodata"]
    transform6 = d["transform"]
    crs_wkt = d["crs_wkt"].item() if np.ndim(d["crs_wkt"]) == 0 else d["crs_wkt"]
    aff = Affine.from_gdal(*transform6)
    src_crs = CRS.from_wkt(crs_wkt)
    return arr, nodata, aff, src_crs

MONTH_MAP = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6,
    "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def _parse_year_month_from_filename(path):
    name = os.path.basename(str(path)).upper()

    m = re.search(r"((?:19|20)\d{2})-((?:19|20)\d{2})-([A-Z]{3,4})(?=[^A-Z]|$)", name)
    if m:
        year = int(m.group(1))
        mon = m.group(3)
        if mon in MONTH_MAP:
            return year, MONTH_MAP[mon]

    m = re.search(r"((?:19|20)\d{2})[-_](0[1-9]|1[0-2])", name)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r"((?:19|20)\d{2})(0[1-9]|1[0-2])", name)
    if m:
        return int(m.group(1)), int(m.group(2))

    return None, None

def sample_monthly_npz_folder_at_stations(
    folder,
    stations_df,
    id_col="naps_id",
    lat_col="lat",
    lon_col="lon",
    npz_glob="*.npz",
    require_same_grid=True,
    # NEW:
    snap_to_nearest_valid=True,
    max_snap_dist_pixels=50,   # limit how far we snap (in pixels). Set None to allow any distance.
    valid_min=None,            # optional: treat values < valid_min as invalid too
    valid_max=None             # optional: treat values > valid_max as invalid too
):
    """
    Returns:
      long_df: [naps_id, lat, lon, label, value, (year, month if parsed)]
      wide_df: one row per station; columns=label (YYYY_MM)

    If sampled value is NaN (nodata/water), optionally snap to nearest valid pixel.
    """

    files = sorted(glob.glob(os.path.join(folder, npz_glob)))
    if not files:
        raise FileNotFoundError(f"No npz files found in: {folder}")

    # Read metadata from first file (assume grid consistent)
    _, _, aff0, crs0 = _read_npz_meta(files[0])
    tfm_to_raster = Transformer.from_crs(CRS.from_epsg(4326), crs0, always_xy=True)

    # Precompute station pixel indices once
    st = stations_df[[id_col, lat_col, lon_col]].copy()
    st[id_col] = st[id_col].astype(str).str.strip()
    st[lat_col] = pd.to_numeric(st[lat_col], errors="coerce")
    st[lon_col] = pd.to_numeric(st[lon_col], errors="coerce")
    st = st.dropna(subset=[lat_col, lon_col]).reset_index(drop=True)

    xs, ys = tfm_to_raster.transform(st[lon_col].to_numpy(), st[lat_col].to_numpy())
    cols_f, rows_f = (~aff0) * (xs, ys)
    st["row"] = np.floor(rows_f).astype(int)
    st["col"] = np.floor(cols_f).astype(int)

    records = []

    for f in files:
        arr, nodata, aff, crs = _read_npz_meta(f)

        if require_same_grid and (crs != crs0 or aff != aff0):
            raise ValueError(
                "Grid differs across files (crs or transform mismatch). "
                f"First file: {files[0]} vs {f}."
            )

        year, month = _parse_year_month_from_filename(f)
        label = os.path.splitext(os.path.basename(f))[0] if year is None else f"{year:04d}_{month:02d}"

        r = st["row"].to_numpy()
        c = st["col"].to_numpy()

        in_bounds = (r >= 0) & (r < arr.shape[0]) & (c >= 0) & (c < arr.shape[1])

        vals = np.full(len(st), np.nan, dtype=float)
        vals[in_bounds] = arr[r[in_bounds], c[in_bounds]].astype(float)

        # mark nodata as NaN
        if nodata is not None:
            vals[vals == nodata] = np.nan

        # optional value range validity
        if valid_min is not None:
            vals[vals < valid_min] = np.nan
        if valid_max is not None:
            vals[vals > valid_max] = np.nan

        # --- NEW: snap NaNs to nearest valid pixel ---
        if snap_to_nearest_valid:
            nan_idx = np.where(np.isnan(vals))[0]
            if nan_idx.size > 0:
                # valid pixel mask
                valid_mask = np.ones(arr.shape, dtype=bool)
                if nodata is not None:
                    valid_mask &= (arr != nodata)
                valid_mask &= ~np.isnan(arr)

                if valid_min is not None:
                    valid_mask &= (arr >= valid_min)
                if valid_max is not None:
                    valid_mask &= (arr <= valid_max)

                valid_rc = np.column_stack(np.where(valid_mask))  # (row, col) for valid pixels

                if valid_rc.size > 0:
                    tree = cKDTree(valid_rc.astype(float))

                    query_rc = np.column_stack([r[nan_idx], c[nan_idx]]).astype(float)
                    dist, nn = tree.query(query_rc, k=1)

                    if max_snap_dist_pixels is not None:
                        ok = dist <= float(max_snap_dist_pixels)
                    else:
                        ok = np.ones_like(dist, dtype=bool)

                    snapped_rows = valid_rc[nn[ok], 0]
                    snapped_cols = valid_rc[nn[ok], 1]
                    snapped_vals = arr[snapped_rows, snapped_cols].astype(float)

                    # still protect against nodata
                    if nodata is not None:
                        snapped_vals[snapped_vals == nodata] = np.nan

                    vals[nan_idx[ok]] = snapped_vals

        out = pd.DataFrame({
            id_col: st[id_col].to_numpy(),
            lat_col: st[lat_col].to_numpy(),
            lon_col: st[lon_col].to_numpy(),
            "label": label,
            "value": vals
        })
        if year is not None:
            out["year"] = year
            out["month"] = month

        records.append(out)

    long_df = pd.concat(records, ignore_index=True)

    # Wide format: keep lat/lon too
    wide_df = (
        long_df.pivot_table(
            index=[id_col, lat_col, lon_col],
            columns="label",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    return long_df, wide_df



In [ ]:
# -----------------------
# Example usage
# -----------------------
long_df, wide_df = sample_monthly_npz_folder_at_stations(
    folder=folder,
    stations_df=stations_df,
    snap_to_nearest_valid=True,
    max_snap_dist_pixels=50  # adjust; 50 pixels might be too large if your grid is fine
)

In [ ]:
long_df

,naps_id,lat,lon,label,value,year,month
0,60104,45.43433,-75.67600,2010_04,2.900000,2010,4
1,60106,45.38287,-75.71387,2010_04,2.900000,2010,4
2,60204,42.31578,-83.04367,2010_04,3.600000,2010,4
3,60211,42.29289,-83.07314,2010_04,3.600000,2010,4
4,60303,44.22008,-76.52141,2010_04,2.500000,2010,4
...,...,...,...,...,...,...,...
5707,61702,43.94594,-78.89492,2023_09,21.500000,2023,9
5708,61703,43.95222,-78.91250,2023_09,21.500000,2023,9
5709,61802,43.55164,-80.26416,2023_09,20.500000,2023,9
5710,65001,44.38231,-79.70243,2023_09,21.799999,2023,9


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations_SATPM25.csv'
long_df.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations_SATPM25.csv


In [ ]:
import rasterio


# 1) Parse year/month from: MYD13A2_NDVI_ON_2023_12.tif
def parse_year_month_from_tif(path):
    name = os.path.basename(str(path))
    m = re.search(r"_((?:19|20)\d{2})_(0[1-9]|1[0-2])\.tif$", name, flags=re.IGNORECASE)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

# 2) Sample all monthly TIFs at stations (lat/lon in EPSG:4326)
def sample_monthly_tifs_at_stations(
    folder,
    stations_df,
    id_col="naps_id",
    lat_col="lat",
    lon_col="lon",
    tif_glob="*.tif",
    band=1
):
    files = sorted(glob.glob(os.path.join(folder, tif_glob)))
    if not files:
        raise FileNotFoundError(f"No tif files found in: {folder}")

    # clean stations
    st = stations_df[[id_col, lat_col, lon_col]].copy()
    st[lat_col] = pd.to_numeric(st[lat_col], errors="coerce")
    st[lon_col] = pd.to_numeric(st[lon_col], errors="coerce")
    st = st.dropna(subset=[lat_col, lon_col]).reset_index(drop=True)

    records = []

    for f in files:
        year, month = parse_year_month_from_tif(f)
        label = os.path.splitext(os.path.basename(f))[0] if year is None else f"{year:04d}_{month:02d}"

        with rasterio.open(f) as src:
            arr = src.read(band)
            nodata = src.nodata
            tfm = src.transform
            crs = src.crs

            if crs is None:
                raise ValueError(f"CRS missing in tif: {f}")

            # lon/lat -> raster CRS x/y
            to_raster = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
            xs, ys = to_raster.transform(st[lon_col].to_numpy(), st[lat_col].to_numpy())

            # x/y -> row/col
            rows, cols = rasterio.transform.rowcol(tfm, xs, ys)
            rows = np.asarray(rows)
            cols = np.asarray(cols)

            vals = np.full(len(st), np.nan, dtype=float)

            in_bounds = (
                (rows >= 0) & (rows < src.height) &
                (cols >= 0) & (cols < src.width)
            )

            vals[in_bounds] = arr[rows[in_bounds], cols[in_bounds]].astype(float)

            # nodata handling
            if nodata is not None:
                vals[vals == nodata] = np.nan

        out = pd.DataFrame({
            id_col: st[id_col].to_numpy(),
            lat_col: st[lat_col].to_numpy(),
            lon_col: st[lon_col].to_numpy(),
            "year": year,
            "month": month,
            "label": label,
            "ndvi": vals
        })
        records.append(out)

    long_df = pd.concat(records, ignore_index=True)

    # station x month wide
    wide_df = (
        long_df.pivot_table(index=id_col, columns="label", values="ndvi", aggfunc="mean")
               .reset_index()
    )

    return long_df, wide_df




In [ ]:
# -----------------------
# Example usage in Colab
# -----------------------
folder_ndvi = "/content/drive/MyDrive/ML project data/NDVI_ON_1km_monthly_2010_2023_Full"

# quick check file list
ndvi_files = sorted(glob.glob(os.path.join(folder_ndvi, "*.tif")))
print("NDVI tif files:", len(ndvi_files))
print("Example:", ndvi_files[:3])

# stations_df should be your unique naps_id/lat/lon table
# stations_df = get_stations_df(wide_by_year)  # if you already created it earlier

long_ndvi, wide_ndvi = sample_monthly_tifs_at_stations(folder_ndvi, stations_df)

display(long_ndvi.head())
display(wide_ndvi.head())

print("NaNs in long_ndvi:", long_ndvi["ndvi"].isna().sum())

NDVI tif files: 168
Example: ['/content/drive/MyDrive/ML project data/NDVI_ON_1km_monthly_2010_2023_Full/MYD13A2_NDVI_ON_2010_01.tif', '/content/drive/MyDrive/ML project data/NDVI_ON_1km_monthly_2010_2023_Full/MYD13A2_NDVI_ON_2010_02.tif', '/content/drive/MyDrive/ML project data/NDVI_ON_1km_monthly_2010_2023_Full/MYD13A2_NDVI_ON_2010_03.tif']


,naps_id,lat,lon,year,month,label,ndvi
0,60104,45.43433,-75.67600,2010,1,2010_01,0.01865
1,60106,45.38287,-75.71387,2010,1,2010_01,-0.00605
2,60204,42.31578,-83.04367,2010,1,2010_01,0.08210
3,60211,42.29289,-83.07314,2010,1,2010_01,0.13490
4,60303,44.22008,-76.52141,2010,1,2010_01,0.05270


label,naps_id,2010_01,2010_02,2010_03,2010_04,2010_05,2010_06,2010_07,2010_08,2010_09,...,2023_03,2023_04,2023_05,2023_06,2023_07,2023_08,2023_09,2023_10,2023_11,2023_12
0,60104,0.01865,0.10235,0.22040,0.4038,0.50885,0.52315,0.50650,0.51725,0.51150,...,0.11205,0.2269,0.44360,0.46300,0.50825,0.49280,0.48085,0.32930,0.26745,0.21405
1,60106,-0.00605,-0.02930,0.26660,0.4676,0.53310,0.60705,0.70555,0.68220,0.63900,...,0.09480,0.3051,0.48135,0.58715,0.69075,0.70735,0.63000,0.48940,0.37325,-0.01110
2,60204,0.08210,0.09535,0.22895,0.3610,0.40705,0.40415,0.39640,0.31625,0.34745,...,0.19795,0.2815,0.35295,0.35230,0.37110,0.37450,0.32875,0.26440,0.23635,0.12060
3,60211,0.13490,0.12335,0.28825,0.4664,0.53595,0.48675,0.49030,0.46810,0.48265,...,0.30940,0.4143,0.40165,0.48965,0.49575,0.55595,0.47840,0.46265,0.38035,0.15630
4,60303,0.05270,0.27975,0.41725,0.5969,0.69265,0.69140,0.77285,0.68960,0.72310,...,0.36585,0.5645,0.62355,0.61425,0.62490,0.68910,0.61210,0.56735,0.51305,0.47295


NaNs in long_ndvi: 0


In [ ]:
wide_ndvi.isna().sum()

,0
label,
naps_id,0
2010_01,0
2010_02,0
2010_03,0
2010_04,0
...,...
2023_08,0
2023_09,0
2023_10,0


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations_ndvi.csv'
long_ndvi.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations_ndvi.csv


In [ ]:
long_ndvi

,naps_id,lat,lon,year,month,label,ndvi
0,60104,45.43433,-75.67600,2010,1,2010_01,0.01865
1,60106,45.38287,-75.71387,2010,1,2010_01,-0.00605
2,60204,42.31578,-83.04367,2010,1,2010_01,0.08210
3,60211,42.29289,-83.07314,2010,1,2010_01,0.13490
4,60303,44.22008,-76.52141,2010,1,2010_01,0.05270
...,...,...,...,...,...,...,...
5707,61702,43.94594,-78.89492,2023,12,2023_12,0.35715
5708,61703,43.95222,-78.91250,2023,12,2023_12,0.25555
5709,61802,43.55164,-80.26416,2023,12,2023_12,0.20515
5710,65001,44.38231,-79.70243,2023,12,2023_12,0.14255


In [ ]:
def sample_single_tif_at_stations(
    tif_path,
    stations_df,
    id_col="naps_id",
    lat_col="lat",
    lon_col="lon",
    band=1,
    out_col="elevation"
):
    st = stations_df[[id_col, lat_col, lon_col]].copy()
    st[lat_col] = pd.to_numeric(st[lat_col], errors="coerce")
    st[lon_col] = pd.to_numeric(st[lon_col], errors="coerce")
    st = st.dropna(subset=[lat_col, lon_col]).reset_index(drop=True)

    with rasterio.open(tif_path) as src:
        arr = src.read(band)
        nodata = src.nodata
        tfm = src.transform
        crs = src.crs

        if crs is None:
            raise ValueError(f"CRS missing in tif: {tif_path}")

        # lon/lat (EPSG:4326) -> raster CRS x/y
        to_raster = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
        xs, ys = to_raster.transform(st[lon_col].to_numpy(), st[lat_col].to_numpy())

        # x/y -> row/col
        rows, cols = rasterio.transform.rowcol(tfm, xs, ys)
        rows = np.asarray(rows)
        cols = np.asarray(cols)

        vals = np.full(len(st), np.nan, dtype=float)

        in_bounds = (
            (rows >= 0) & (rows < src.height) &
            (cols >= 0) & (cols < src.width)
        )

        vals[in_bounds] = arr[rows[in_bounds], cols[in_bounds]].astype(float)

        if nodata is not None:
            vals[vals == nodata] = np.nan

    out = st.copy()
    out["row"] = rows
    out["col"] = cols
    out[out_col] = vals
    return out

In [ ]:
folder_elev = "/content/drive/MyDrive/ML project data/GMTED2010_Ontario"
tif_elev = os.path.join(folder_elev, "GMTED2010_min_elevation_Ontario_bbox.tif")

stations_elev = sample_single_tif_at_stations(
    tif_path=tif_elev,
    stations_df=stations_df,
    out_col="elev_m"   # meters (likely)
)

display(stations_elev.head())
print("NaNs:", stations_elev["elev_m"].isna().sum())

,naps_id,lat,lon,row,col,elev_m
0,60104,45.43433,-75.67600,1377,2345,54.0
1,60106,45.38287,-75.71387,1383,2341,79.0
2,60204,42.31578,-83.04367,1752,1460,182.0
3,60211,42.29289,-83.07314,1755,1457,177.0
4,60303,44.22008,-76.52141,1523,2244,93.0


NaNs: 0


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations_elevation.csv'
stations_elev.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations_elevation.csv


In [ ]:
def sample_categorical_tif_at_stations(
    tif_path,
    stations_df,
    id_col="naps_id",
    lat_col="lat",
    lon_col="lon",
    out_col="landcover"
):
    st = stations_df[[id_col, lat_col, lon_col]].copy()
    st[lat_col] = pd.to_numeric(st[lat_col], errors="coerce")
    st[lon_col] = pd.to_numeric(st[lon_col], errors="coerce")
    st = st.dropna(subset=[lat_col, lon_col]).reset_index(drop=True)

    with rasterio.open(tif_path) as src:
        if src.crs is None:
            raise ValueError(f"Missing CRS: {tif_path}")

        # lon/lat -> raster CRS
        to_raster = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xs, ys = to_raster.transform(st[lon_col].to_numpy(), st[lat_col].to_numpy())
        coords = list(zip(xs, ys))

        # stream sample (no full read)
        vals = np.array([v[0] for v in src.sample(coords, indexes=1)])

        nodata = src.nodata
        if nodata is not None:
            vals = vals.astype("float")
            vals[vals == nodata] = np.nan

    out = st.copy()
    out[out_col] = vals
    return out



In [ ]:
lc_paths = {
    2010: "/content/drive/MyDrive/ML project data/LandCover_Ontario_5km/LandCover_ON_2010_5km_mode.tif",
    2015: "/content/drive/MyDrive/ML project data/LandCover_Ontario_5km/LandCover_ON_2015_5km_mode.tif",
    2020: "/content/drive/MyDrive/ML project data/LandCover_Ontario_5km/LandCover_ON_2020_5km_mode.tif",
}

out = stations_df.copy()
for y, p in lc_paths.items():
    tmp = sample_categorical_tif_at_stations(p, stations_df, out_col=f"lc_{y}")
    out = out.merge(tmp[["naps_id", f"lc_{y}"]], on="naps_id", how="left")

display(out.head())

,naps_id,lat,lon,lc_2010,lc_2015,lc_2020
0,60104,45.43433,-75.67600,17.0,17.0,17.0
1,60106,45.38287,-75.71387,17.0,17.0,17.0
2,60204,42.31578,-83.04367,17.0,17.0,17.0
3,60211,42.29289,-83.07314,18.0,18.0,18.0
4,60303,44.22008,-76.52141,17.0,17.0,17.0


In [ ]:
output_path = '/content/drive/MyDrive/ML project data/stations_landcover.csv'
out.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: /content/drive/MyDrive/ML project data/stations_landcover.csv
